In [1]:

import os

os.makedirs("data", exist_ok=True)

produits = [
    (1, "Laptop", "Informatique", 8500, 5),
    (2, "Souris", "Informatique", 150, 40),
    (3, "Clavier", "Informatique", 300, 0),
    (4, "Chaise", "Mobilier", 900, 12),
    (5, "Bureau", "Mobilier", 2200, 3),
    (6, "Stylo", "Papeterie", 5, 300),
    (7, "Cahier", "Papeterie", 15, 120),
    (8, "Écran", "Informatique", 2500, 8),
]

with open("data/produits.csv", "w", encoding="utf-8") as f:
    f.write("id_produit,nom,categorie,prix,stock\n")
    for t in produits:
        f.write(",".join(str(x) for x in t) + "\n")

print(os.listdir("data"))


['.ipynb_checkpoints', 'produits.csv', 'ventes.csv']


In [5]:
from pyspark.sql import SparkSession 
spark = SparkSession.builder.appName("Boutique").master("local[*]").getOrCreate()
sc = spark.sparkContext
print("Session crée avec succés")
print(sc.version)


Session crée avec succés
4.0.0


In [6]:
rdd_lignes = sc.textFile("data/produits.csv")
rdd_lignes.collect()

['id_produit,nom,categorie,prix,stock',
 '1,Laptop,Informatique,8500,5',
 '2,Souris,Informatique,150,40',
 '3,Clavier,Informatique,300,0',
 '4,Chaise,Mobilier,900,12',
 '5,Bureau,Mobilier,2200,3',
 '6,Stylo,Papeterie,5,300',
 '7,Cahier,Papeterie,15,120',
 '8,Écran,Informatique,2500,8']

In [8]:

rdd_lignes = sc.textFile("data/produits.csv")

rdd_prod = rdd_lignes \
    .filter(lambda x: not x.startswith("id_produit")) \
    .map(lambda x: x.split(",")) \
    .map(lambda x: (int(x[0]), x[1], x[2], int(x[3]), int(x[4])))


In [9]:
rdd_prod.take(3)

[(1, 'Laptop', 'Informatique', 8500, 5),
 (2, 'Souris', 'Informatique', 150, 40),
 (3, 'Clavier', 'Informatique', 300, 0)]

In [16]:
rdd_prod.filter(lambda x: x[4] < 10).map(lambda x : x[1]).collect()

['Laptop', 'Clavier', 'Bureau', 'Écran']

In [20]:
df_prod = spark.read.option("header",True).option("inferSchema",True).csv("data/produits.csv")

df_prod.show()
df_prod.printSchema()

+----------+-------+------------+----+-----+
|id_produit|    nom|   categorie|prix|stock|
+----------+-------+------------+----+-----+
|         1| Laptop|Informatique|8500|    5|
|         2| Souris|Informatique| 150|   40|
|         3|Clavier|Informatique| 300|    0|
|         4| Chaise|    Mobilier| 900|   12|
|         5| Bureau|    Mobilier|2200|    3|
|         6|  Stylo|   Papeterie|   5|  300|
|         7| Cahier|   Papeterie|  15|  120|
|         8|  Écran|Informatique|2500|    8|
+----------+-------+------------+----+-----+

root
 |-- id_produit: integer (nullable = true)
 |-- nom: string (nullable = true)
 |-- categorie: string (nullable = true)
 |-- prix: integer (nullable = true)
 |-- stock: integer (nullable = true)



In [28]:
from pyspark.sql import functions as F

df_prod.filter(
    (F.col("categorie") == "Informatique") &
    (F.col("prix") > 200)
).select("nom", "prix").orderBy(F.col("prix").desc()).show()
    

+-------+----+
|    nom|prix|
+-------+----+
| Laptop|8500|
|  Écran|2500|
|Clavier| 300|
+-------+----+



In [29]:
df_prod = df_prod.withColumn(
    "status",
    F.when(F.col("stock") == 0 , "rupture de stock")
    .when(F.col("stock") < 10 , "faible")
    .otherwise("ok")
)
df_prod.show()

+----------+-------+------------+----+-----+----------------+
|id_produit|    nom|   categorie|prix|stock|          status|
+----------+-------+------------+----+-----+----------------+
|         1| Laptop|Informatique|8500|    5|          faible|
|         2| Souris|Informatique| 150|   40|              ok|
|         3|Clavier|Informatique| 300|    0|rupture de stock|
|         4| Chaise|    Mobilier| 900|   12|              ok|
|         5| Bureau|    Mobilier|2200|    3|          faible|
|         6|  Stylo|   Papeterie|   5|  300|              ok|
|         7| Cahier|   Papeterie|  15|  120|              ok|
|         8|  Écran|Informatique|2500|    8|          faible|
+----------+-------+------------+----+-----+----------------+



In [31]:
df_prod.createOrReplaceTempView("produits")


In [34]:

req1 = spark.sql("""
    SELECT
        categorie,
        COUNT(*) AS nbr_produits,
        MAX(prix) AS max_prix
    FROM produits
    GROUP BY categorie
    HAVING MAX(prix) > 1000
    ORDER BY max_prix DESC
""")

req1.show()


+------------+------------+--------+
|   categorie|nbr_produits|max_prix|
+------------+------------+--------+
|Informatique|           4|    8500|
|    Mobilier|           2|    2200|
+------------+------------+--------+

